In [1]:
import sys
import os
sys.path.append(os.path.abspath('../..'))
import torch
import torch.nn.functional as F
import src.models.v4.nn as nn
from src.models.v4.tokenizer import Tokenizer
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data_path = "../../src/data/normalized/en_cities.txt"

tokenizer = Tokenizer(data_path)

In [3]:
context_length = 4
embedding_size = 16
w1_out = 32
w2_out = len(tokenizer)

batch_size = 32

In [4]:
# Dataset
x = []
raw_out = []

with open(data_path) as f:
    for word in f.readlines():
        word = word.strip()
        context = [0] * context_length
        for letter in word:
            ordinal = tokenizer.stoi[letter]
            x.append(context)
            raw_out.append(ordinal)
            
            context = context[1:] + [ordinal]
        x.append(context)
        raw_out.append(0)

In [5]:
# Split data into training, dev and evaluation sets
d1 = int(len(x) * 0.8)
d2 = int(len(x) * 0.9)

x_tra = torch.tensor(x[:d1])
y_tra = torch.tensor(raw_out[:d1])

x_dev = torch.tensor(x[d1:d2])
y_dev = torch.tensor(raw_out[d1:d2])

x_eval = torch.tensor(x[d2:])
y_eval = torch.tensor(raw_out[d2:])

In [6]:
# Network Layers
embedding = torch.randn(len(tokenizer), embedding_size)

layers = [
    nn.Linear(context_length * embedding_size, w1_out, has_bias=False),
    nn.BatchNorm1d(w1_out),
    nn.Tanh(),
    nn.Linear(w1_out, w2_out)
]

In [7]:
# Get parameters function
def parameters() -> list[torch.Tensor]:
    p = [embedding]
    for layer in layers:
        p.extend(layer.parameters())
    return p

In [8]:
# Enable requires_grad in parameters
def set_gradients(enable: bool):
    for parameter in parameters():
        parameter.requires_grad = enable

In [9]:
def set_training(enable: bool):
    set_gradients(enable)
    batchnorm_layers = [layer for layer in layers if isinstance(layer, nn.BatchNorm1d)]
    for layer in batchnorm_layers:
        layer.training = enable

In [10]:
# Forward pass function

def forward(x: torch.Tensor, batch_size=batch_size) -> torch.Tensor:
    
    out = [embedding[x].view(batch_size, -1)] # emb: (tokens, emb) [ (batch, context) ] -> (batch, context, emb)
    
    for layer in layers:
        out.append(layer(out[-1]))
    
    return out[-1]

In [11]:
i = 0

In [12]:
iterations = 30_000
learning_rate = 0.005

In [13]:
set_training(True)
for _ in range(iterations):
    i += 1 # type: ignore
    
    # 0. Get batch
    indexes = torch.randint(0, len(x_tra), (batch_size,))
    batch_x = x_tra[indexes]
    batch_y = y_tra[indexes]
    
    # 1. Forward pass
    logits = forward(batch_x) # (batch, tokens)
    
    #   0.4     -1.5     0.1     -1       2.5
    #   1       0.1      2.3     -0.6     -1.2
    #   
    #   1.5     0.15     0.4    0.3      30
    #   3       1.2      10     0.7     0.2
    #   
    
    counts = logits.exp() # (batch, tokens)
    probablities = counts / counts.sum(dim=1, keepdim=True) # (batch, tokens)
    # i know that if some of the raw outputs are large enough, it will become
    # a +inf, thus the sum too, or if they are all 0, the sum will be 0,
    # but it won't really matter for this setup, and I want to make these things
    # by hand, not just use the torch cross entropy loss
    
    # 2. Get cross entropy loss
    correct_probablities = probablities[torch.arange(batch_size), batch_y] # (batch)
    
    # we want this to all be as close to 1 as possible,
    # so basically we want their product to be as close
    # to 1 as possible.
    # thus, using the product rule of logarithms,
    # we need their logarithms to be as close to zero 
    # as possible. but because they are all nonpositive,
    # we take the negation, and try to minimise that
    # with the model.
    
    log_mean_likelihood = correct_probablities.log().mean()
    loss = -log_mean_likelihood
    
    #loss = F.cross_entropy(logits, batch_y)
    
    if i % 500 == 0 or i==1:
        print(f'{i}.\tloss: {loss}')
    
    # 3. Zero gradients
    for param in parameters():
        param.grad = None
    
    # 4. Backward pass
    loss.backward()
    
    for param in parameters():
        param.data -= learning_rate * param.grad # type: ignore

1.	loss: 3.5139458179473877
500.	loss: 3.297452449798584
1000.	loss: 3.124310255050659
1500.	loss: 2.963136672973633
2000.	loss: 2.8650763034820557
2500.	loss: 2.760584592819214
3000.	loss: 2.8815736770629883
3500.	loss: 2.8199684619903564
4000.	loss: 2.7243223190307617
4500.	loss: 2.762878179550171
5000.	loss: 2.8469455242156982
5500.	loss: 2.779768943786621
6000.	loss: 2.7166051864624023
6500.	loss: 2.9762768745422363
7000.	loss: 2.391115665435791
7500.	loss: 1.9717274904251099
8000.	loss: 2.4365272521972656
8500.	loss: 3.069627285003662
9000.	loss: 2.6390926837921143
9500.	loss: 2.062204360961914
10000.	loss: 2.348909378051758
10500.	loss: 2.2389421463012695
11000.	loss: 2.331408977508545
11500.	loss: 2.5778729915618896
12000.	loss: 2.573939561843872
12500.	loss: 2.33979868888855
13000.	loss: 2.535130500793457
13500.	loss: 2.463531494140625
14000.	loss: 2.508484125137329
14500.	loss: 2.5723376274108887
15000.	loss: 2.718259334564209
15500.	loss: 2.679192304611206
16000.	loss: 2.0983

In [14]:
# Inference

def inference(context: list[int]) -> int:
    logits = forward(torch.tensor(context).int(), batch_size=1).view((len(tokenizer),))
    
    counts = torch.exp(logits)
    probablities = counts / counts.sum()
    return int(torch.multinomial(probablities, num_samples=1, replacement=True).item())

In [15]:
def sample() -> str:
    context = [0] * context_length
    generated = [-1]
    while generated[-1] != 0:
        y = inference(context)
        generated.append(y)
        context = context[1:] + [y]
    return tokenizer.decode(generated[1:-1])

In [38]:
set_training(False)

sample()

'labloch'